# Facial Recognition Technology (FRT) Ethics Prototype


The aim is not to build a fully operational facial recognition system, but to:
- demonstrate how the system behaves under different configurations,
- apply both functional and ethical constraints,
- and identify situations where the system should be prevented from operating.

The implementation follows the variables defined in the report:

- **T**: face-match threshold  
- **N**: watchlist size  
- **R**: number of candidates returned  
- **Csev**: crime severity level  

The focus is on decision logic, validation, and transparency rather than model accuracy.

In [25]:
from dataclasses import dataclass
from typing import List, Dict, Any, Optional
from datetime import datetime
from pprint import pprint
import copy

## 1. System Setup

This section defines a small synthetic dataset used for testing.

It includes:
- a watchlist of individuals,
- a set of candidate matches returned by the system,
- and a simplified FPIR table to represent fairness risk across demographic groups.

The values are illustrative and are used to test system behaviour rather than reflect real-world performance.

In [26]:
# ---------------------------
# WATCHLIST DATA
# ---------------------------

watchlist = [
    {"person_id": "W001", "name": "Aoife Byrne", "reason": "Outstanding warrant", "risk_tag": "wanted"},
    {"person_id": "W002", "name": "Mohammed Rahman", "reason": "Missing vulnerable person", "risk_tag": "missing"},
    {"person_id": "W003", "name": "Conor Clarke", "reason": "Breach of court order", "risk_tag": "wanted"},
    {"person_id": "W004", "name": "Siobhan Murphy", "reason": "Outstanding warrant", "risk_tag": "wanted"},
    {"person_id": "W005", "name": "Arjun Patel", "reason": "Missing vulnerable person", "risk_tag": "missing"},
]

# ---------------------------
# OBSERVED CANDIDATES
# ---------------------------

observed_candidates = [
    {
        "candidate_id": "W001",
        "display_name": "Aoife Byrne",
        "similarity_score": 0.93,
        "demographic_group": "White",
        "image_quality": "good",
        "occlusion": False
    },
    {
        "candidate_id": "W003",
        "display_name": "Kwame Mensah",
        "similarity_score": 0.84,
        "demographic_group": "Black",
        "image_quality": "medium",
        "occlusion": False
    },
    {
        "candidate_id": "W005",
        "display_name": "Arjun Patel",
        "similarity_score": 0.79,
        "demographic_group": "Asian",
        "image_quality": "medium",
        "occlusion": True
    },
    {
        "candidate_id": "W999",
        "display_name": "Unknown individual",
        "similarity_score": 0.67,
        "demographic_group": "Black",
        "image_quality": "poor",
        "occlusion": True
    },
]

# ---------------------------
# FPIR TABLE
# ---------------------------

fpir_table = {
    0.90: {"White": 0.002, "Black": 0.003, "Asian": 0.003},
    0.85: {"White": 0.004, "Black": 0.007, "Asian": 0.006},
    0.80: {"White": 0.009, "Black": 0.017, "Asian": 0.015},
    0.75: {"White": 0.018, "Black": 0.036, "Asian": 0.031},
    0.70: {"White": 0.031, "Black": 0.064, "Asian": 0.058},
    0.65: {"White": 0.050, "Black": 0.101, "Asian": 0.091},
    0.60: {"White": 0.079, "Black": 0.152, "Asian": 0.139},
}

## 2. Functional Behaviour

This section implements the basic search functionality.

The system:
- filters candidates using threshold **T**,
- sorts results by similarity score,
- and returns up to **R** candidates.

This allows us to validate that the system behaves correctly before introducing ethical constraints.

In [27]:
def normalise_threshold(threshold: float) -> float:
    available = sorted(fpir_table.keys())
    return min(available, key=lambda x: abs(x - threshold))


def run_search(candidates: List[Dict[str, Any]], threshold: float, return_limit: int) -> List[Dict[str, Any]]:
    threshold = normalise_threshold(threshold)

    results = [c for c in candidates if c["similarity_score"] >= threshold]
    results = sorted(results, key=lambda x: x["similarity_score"], reverse=True)

    return results[:return_limit]


def print_search_results(results: List[Dict[str, Any]]) -> None:
    if not results:
        print("No candidates returned.")
        return

    for i, r in enumerate(results, 1):
        print(
            f"{i}. {r['display_name']} "
            f"(score={r['similarity_score']:.2f}, "
            f"group={r['demographic_group']}, "
            f"quality={r['image_quality']})"
        )

## 3. Fairness Constraints

This section introduces a fail-safe mechanism.

The system evaluates:
- whether the threshold is too low,
- whether demographic disparity is excessive,
- and whether the watchlist size is proportionate.

If these conditions are not met, the system prevents the search from proceeding.

In [28]:
def get_fpir_profile(threshold: float):
    return fpir_table[normalise_threshold(threshold)]


def compute_disparity(fpir_profile: Dict[str, float]) -> float:
    return max(fpir_profile.values()) - min(fpir_profile.values())


def assess_watchlist_pressure(size: int) -> str:
    if size <= 5:
        return "low"
    elif size <= 20:
        return "moderate"
    return "high"


def check_fairness_constraints(threshold: float, watchlist_size: int):
    threshold = normalise_threshold(threshold)
    profile = get_fpir_profile(threshold)
    disparity = compute_disparity(profile)
    pressure = assess_watchlist_pressure(watchlist_size)

    passed = True
    warnings = []
    blocking_reasons = []

    if threshold < 0.60:
        passed = False
        blocking_reasons.append("Threshold below acceptable minimum.")

    if threshold <= 0.80 and disparity > 0.006:
        warnings.append("Increased demographic disparity detected.")

    if pressure == "high" and threshold <= 0.80:
        passed = False
        blocking_reasons.append("Watchlist size is disproportionate for this threshold.")

    if threshold <= 0.75 and disparity >= 0.015:
        passed = False
        blocking_reasons.append("Configuration presents elevated risk of false identification.")

    if not warnings and not blocking_reasons:
        warnings.append("Fairness constraints satisfied.")

    return {
        "passed": passed,
        "threshold": threshold,
        "watchlist_size": watchlist_size,
        "disparity": round(disparity, 3),
        "warnings": warnings,
        "blocking_reasons": blocking_reasons
    }

## 4. Severity-Based Adjustment

The system adjusts its configuration based on crime severity (**Csev**).

Higher severity allows broader search parameters.
However, these proposals must still pass fairness constraints.

In [29]:
def adjust_for_severity(Csev: int):
    if Csev == 1:
        return {"threshold": 0.90, "return_limit": 1}
    elif Csev == 2:
        return {"threshold": 0.85, "return_limit": 2}
    elif Csev == 3:
        return {"threshold": 0.80, "return_limit": 3}
    elif Csev == 4:
        return {"threshold": 0.75, "return_limit": 5}
    elif Csev == 5:
        return {"threshold": 0.60, "return_limit": 10}
    else:
        raise ValueError("Csev must be an integer between 1 and 5.")

## 5. Human Oversight and Privacy

System outputs are not treated as final decisions.

A human reviewer must:
- approve or reject candidates,
- and ensure appropriate action is taken.

Non-matching biometric data is removed to support data minimisation.

In [30]:
def review_candidate(candidate, decision, note="Manual review completed."):
    return {
        "candidate": candidate["display_name"],
        "decision": decision,
        "note": note,
        "time": datetime.now().isoformat(timespec="seconds")
    }


def delete_non_matches(results, approved_id):
    return [r["candidate_id"] for r in results if r["candidate_id"] != approved_id]

## 6. Integrated System

This function combines all components of the prototype.

The process is as follows:
1. A configuration is proposed based on severity (**Csev**)
2. Fairness constraints are evaluated
3. The search is either blocked or executed
4. Results are subject to human review
5. Non-matching data is removed
6. All steps are recorded in an audit log

This allows both functional and ethical behaviour to be evaluated together.

In [31]:
def full_system(
    candidates: List[Dict[str, Any]],
    watchlist: List[Dict[str, Any]],
    Csev: int,
    reviewer_decision: str = "reject"
) -> Dict[str, Any]:

    audit_log = []

    # Step 1: severity-based configuration
    config = adjust_for_severity(Csev)
    audit_log.append({"event": "config_proposed", "data": config})

    # Step 2: fairness check
    fairness = check_fairness_constraints(config["threshold"], len(watchlist))
    audit_log.append({"event": "fairness_check", "data": fairness})

    if not fairness["passed"]:
        return {
            "status": "BLOCKED",
            "reason": "Search prevented due to fairness constraints.",
            "config": config,
            "fairness": fairness,
            "results": [],
            "review": None,
            "deleted": [],
            "audit": audit_log
        }

    # Step 3: run search
    results = run_search(candidates, config["threshold"], config["return_limit"])
    audit_log.append({"event": "search_run", "count": len(results)})

    if not results:
        return {
            "status": "NO_MATCH",
            "reason": "No candidates met the threshold.",
            "config": config,
            "fairness": fairness,
            "results": [],
            "review": None,
            "deleted": [],
            "audit": audit_log
        }

    # Step 4: human review (top result only)
    top = results[0]
    review = review_candidate(top, reviewer_decision)
    audit_log.append({"event": "human_review", "data": review})

    approved_id = top["candidate_id"] if reviewer_decision == "approve" else None

    # Step 5: delete non-matches
    deleted = delete_non_matches(results, approved_id)
    audit_log.append({"event": "cleanup", "deleted": deleted})

    # Step 6: final decision
    if reviewer_decision == "approve":
        status = "ACTION_TAKEN"
        reason = "Candidate approved following review."
    else:
        status = "REJECTED_AFTER_REVIEW"
        reason = "Candidate not approved by reviewer."

    return {
        "status": status,
        "reason": reason,
        "config": config,
        "fairness": fairness,
        "results": results,
        "review": review,
        "deleted": deleted,
        "audit": audit_log
    }

## 7. Output Formatting

This helper function presents system outputs in a structured format.

It is primarily used to support readability and provide clear evidence for validation.

In [32]:
def print_decision_summary(decision):
    print("=" * 60)
    print(f"STATUS: {decision['status']}")
    print(f"REASON: {decision['reason']}")
    print("-" * 60)

    print("\nConfiguration:")
    pprint(decision["config"])

    print("\nFairness:")
    pprint(decision["fairness"])

    print("\nResults:")
    print_search_results(decision["results"])

    print("\nHuman Review:")
    pprint(decision["review"])

    print("\nDeleted Data:")
    pprint(decision["deleted"])

    print("\nAudit Log:")
    for entry in decision["audit"]:
        print(f"- {entry['event']}")

    print("=" * 60)

## 8. Validation Scenarios

The following scenarios are used to evaluate system behaviour.

They demonstrate:
- correct functional operation,
- the effect of severity adjustments,
- the role of human review,
- and situations where the system is prevented from operating.

In [33]:
# Scenario 1: Low severity, strict threshold
scenario_1 = full_system(
    candidates=observed_candidates,
    watchlist=watchlist,
    Csev=1,
    reviewer_decision="approve"
)

print("Scenario 1: Low severity")
print_decision_summary(scenario_1)

Scenario 1: Low severity
STATUS: ACTION_TAKEN
REASON: Candidate approved following review.
------------------------------------------------------------

Configuration:
{'return_limit': 1, 'threshold': 0.9}

Fairness:
{'blocking_reasons': [],
 'disparity': 0.001,
 'passed': True,
 'threshold': 0.9,
 'warnings': ['Fairness constraints satisfied.'],
 'watchlist_size': 5}

Results:
1. Aoife Byrne (score=0.93, group=White, quality=good)

Human Review:
{'candidate': 'Aoife Byrne',
 'decision': 'approve',
 'note': 'Manual review completed.',
 'time': '2026-03-19T23:26:36'}

Deleted Data:
[]

Audit Log:
- config_proposed
- fairness_check
- search_run
- human_review
- cleanup


In [34]:
# Scenario 2: Moderate severity, human rejects result
scenario_2 = full_system(
    candidates=observed_candidates,
    watchlist=watchlist,
    Csev=3,
    reviewer_decision="reject"
)

print("Scenario 2: Moderate severity")
print_decision_summary(scenario_2)

Scenario 2: Moderate severity
STATUS: REJECTED_AFTER_REVIEW
REASON: Candidate not approved by reviewer.
------------------------------------------------------------

Configuration:
{'return_limit': 3, 'threshold': 0.8}

Fairness:
{'blocking_reasons': [],
 'disparity': 0.008,
 'passed': True,
 'threshold': 0.8,
 'warnings': ['Increased demographic disparity detected.'],
 'watchlist_size': 5}

Results:
1. Aoife Byrne (score=0.93, group=White, quality=good)
2. Kwame Mensah (score=0.84, group=Black, quality=medium)

Human Review:
{'candidate': 'Aoife Byrne',
 'decision': 'reject',
 'note': 'Manual review completed.',
 'time': '2026-03-19T23:26:36'}

Deleted Data:
['W001', 'W003']

Audit Log:
- config_proposed
- fairness_check
- search_run
- human_review
- cleanup


In [35]:
# Scenario 3: High severity, broader search
scenario_3 = full_system(
    candidates=observed_candidates,
    watchlist=watchlist,
    Csev=5,
    reviewer_decision="reject"
)

print("Scenario 3: High severity blocked by fairness constraints")
print_decision_summary(scenario_3)

Scenario 3: High severity blocked by fairness constraints
STATUS: BLOCKED
REASON: Search prevented due to fairness constraints.
------------------------------------------------------------

Configuration:
{'return_limit': 10, 'threshold': 0.6}

Fairness:
{'blocking_reasons': ['Configuration presents elevated risk of false '
                      'identification.'],
 'disparity': 0.073,
 'passed': False,
 'threshold': 0.6,
 'warnings': ['Increased demographic disparity detected.'],
 'watchlist_size': 5}

Results:
No candidates returned.

Human Review:
None

Deleted Data:
[]

Audit Log:
- config_proposed
- fairness_check


In [36]:
# Scenario 4: Large watchlist triggers blocking

large_watchlist = watchlist + [
    {"person_id": f"W{100+i}", "name": f"Additional Person {i}", "reason": "Expanded list", "risk_tag": "wanted"}
    for i in range(30)
]

scenario_4 = full_system(
    candidates=observed_candidates,
    watchlist=large_watchlist,
    Csev=5,
    reviewer_decision="approve"
)

print("Scenario 4: Large watchlist (blocked)")
print_decision_summary(scenario_4)

Scenario 4: Large watchlist (blocked)
STATUS: BLOCKED
REASON: Search prevented due to fairness constraints.
------------------------------------------------------------

Configuration:
{'return_limit': 10, 'threshold': 0.6}

Fairness:
{'blocking_reasons': ['Watchlist size is disproportionate for this threshold.',
                      'Configuration presents elevated risk of false '
                      'identification.'],
 'disparity': 0.073,
 'passed': False,
 'threshold': 0.6,
 'warnings': ['Increased demographic disparity detected.'],
 'watchlist_size': 35}

Results:
No candidates returned.

Human Review:
None

Deleted Data:
[]

Audit Log:
- config_proposed
- fairness_check


## 9. Summary

This implementation shows:

1. Functional performance alone is insufficient.  
2. Severity-based adjustments must remain constrained.  
3. Human oversight and data minimisation are essential safeguards.  